In [1]:
#installs
!pip install --upgrade --quiet  langchain_google_community
!pip install huggingface_hub
!pip install --upgrade langchain
!pip install langchain-community langchain-core
!pip install -U langchain-huggingface
!pip install --upgrade gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.4/84.4 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 19.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.6/49.6 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.6/57.6 MB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 321.4/321.4 kB 23.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.8/94.8 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 70.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.2/73.2 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.3/62.3 kB 4.8 MB/s eta 0:00:00
  Attempting uninstall: markupsafe
    Found existing installation: MarkupSafe 3.0.2
    Uninstalling MarkupSafe-3.0.2:
      Successfully uninstalled MarkupSafe-3.0.2


GOOGLE CUSTOM SEARCH TOOL

In [23]:
#IMPORTS
import os
from langchain_core.tools import Tool
from langchain_google_community import GoogleSearchAPIWrapper
from google.colab import userdata

In [25]:
#ENVIRONMENT SETUP
os.environ["GOOGLE_CSE_ID"] = userdata.get('GOOGLE_CSE_ID')
os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')

#CREATING GOOGLE SEARCH TOOL
search = GoogleSearchAPIWrapper()

tool = Tool(
    name="google_search",
    description="Search Google for recent results.",
    func=search.run,
)

MODEL

In [4]:
#IMPORTS
import torch
from huggingface_hub import login
from langchain_community.llms import HuggingFaceEndpoint
from langchain_community.chat_models.huggingface import ChatHuggingFace

In [28]:
#LOGIN
login(token=userdata.get('HGtoken'))

#CREATING MODEL WITH ENDPOINT
llm=HuggingFaceEndpoint(repo_id="HuggingFaceH4/zephyr-7b-beta")
local_llm=ChatHuggingFace(llm=llm)

EMBEDDING AGENT

In [6]:
#IMPORTS
from langchain.agents import initialize_agent, load_tools, AgentExecutor, create_structured_chat_agent
from langchain import hub
from langchain.memory import ChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain import PromptTemplate

In [7]:
#PROMPT FOR AGENT
prompt = hub.pull("hwchase17/structured-chat-agent")


#CHAT MEMORY STORAGE
memory = ChatMessageHistory(session_id="session")

#AGENT
agent=create_structured_chat_agent(local_llm,
                                   [tool],
                                   prompt
                                   )
#AGENT_EXECUTOR
agent_executor = AgentExecutor(agent=agent, tools=[tool],
                               handle_parsing_errors=True,
                               max_iterations=10
                               )

#EXECUTABLE AGENT WITH MEMORY
agent_with_chat_history = RunnableWithMessageHistory(agent_executor,
                                                     lambda session_id: memory,
                                                     input_messages_key="input",
                                                     history_messages_key="chat_history"
                                                    )

/usr/local/lib/python3.10/dist-packages/langsmith/client.py:256: LangSmithMissingAPIKeyWarning: API key must be provided when using hosted LangSmith API
  warnings.warn(


USER INTERFACE

In [8]:
#imports
import gradio as gr

In [9]:
#Chatbot conversation function
def chatbot_response_conversation(message, history):
  answer = agent_with_chat_history.invoke({"input":message},
                                          config={"configurable": {"session_id": "<foo>"}})
  return answer['output']

In [16]:
#chatbot UI
travel_agent = chatbot_conversation_ui = gr.ChatInterface(chatbot_response_conversation,
                              title="Travel Agent")

/usr/local/lib/python3.10/dist-packages/gradio/components/chatbot.py:279: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  warnings.warn(


In [17]:
# Launch the app
travel_agent.launch(inbrowser=True, share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://6cdff1407e26781199.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
